In [1]:
from qdrant_client import AsyncQdrantClient, models
from langchain_huggingface import HuggingFaceEmbeddings
from rag_modules import JiebaFastEmbed, RetrievalOptimizationModule, Qdrant, DataPreparationModule
from qdrant_client.models import PointStruct, SparseVector, Prefetch, FusionQuery, Fusion

/opt/conda/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
data_module = DataPreparationModule("../../data/C8/cook")
data_module.load_documents()
data_module.chunk_documents();

{'主标题': '小酥肉的做法', 'source': '../../data/C8/cook/dishes/meat_dish/小酥肉.md', 'parent_id': 'a8d62aca1273a4da20cab6ab4cf25c1d', 'doc_type': 'child', 'category': '荤菜', 'dish_name': '小酥肉', 'difficulty': '中等', 'chunk_id': 'b365a599-d34d-41a1-b93b-f1e24667af9f', 'chunk_index': 0}


In [4]:
data_module.chunks[300].metadata

{'主标题': '蒸花卷的做法',
 '二级标题': '必备原料和工具',
 'source': '../../data/C8/cook/dishes/breakfast/蒸花卷.md',
 'parent_id': '072d76a49674e8ca5999ecc26af75266',
 'doc_type': 'child',
 'category': '早餐',
 'dish_name': '蒸花卷',
 'difficulty': '简单',
 'chunk_id': '647a94d672a1ad33c88e0ecca78bee52',
 'chunk_index': 1,
 'batch_index': 300,
 'chunk_size': 40}

In [3]:
data_module.chunks[300].metadata

{'主标题': '蒸花卷的做法',
 '二级标题': '必备原料和工具',
 'source': '../../data/C8/cook/dishes/breakfast/蒸花卷.md',
 'parent_id': '072d76a49674e8ca5999ecc26af75266',
 'doc_type': 'child',
 'category': '早餐',
 'dish_name': '蒸花卷',
 'difficulty': '简单',
 'chunk_id': '647a94d672a1ad33c88e0ecca78bee52',
 'chunk_index': 1,
 'batch_index': 300,
 'chunk_size': 40}

# 向量数据库初始化

In [6]:
qdrant_client = AsyncQdrantClient(url="http://localhost:6334", check_compatibility=False, prefer_grpc=True)

In [7]:
chunk_collection_name = "RecipeChunk"
if not await qdrant_client.collection_exists("RecipeChunk"):
    await qdrant_client.create_collection(
        collection_name=chunk_collection_name, 
        vectors_config={'dense': models.VectorParams(
            size=1024, 
            distance=models.Distance.COSINE, 
            on_disk=True,  # 稠密向量是否在内存，可以节省内存，牺牲时间
            datatype=models.Datatype.FLOAT32,  # 默认32,16节省内存
        )}, 
        sparse_vectors_config={'sparse': models.SparseVectorParams(
            index=models.SparseIndexParams(
                on_disk=True,  # 内存优化：稀疏索引放硬盘
                datatype=models.Datatype.FLOAT32
            )
        )}, 
        hnsw_config=models.models.HnswConfigDiff(on_disk=True), 
        on_disk_payload=False  # 元数据直接存在内存中，元数据不大
    )

In [6]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
dense_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)
sparse_model = JiebaFastEmbed(model_name='Qdrant/bm25')  # 重新封装的适合中文的稀疏向量模型

2026-04-05 05:11:56.929 | ERROR    | fastembed.common.model_management:download_model:446 - Could not download model from HuggingFace: 429 Client Error: Too Many Requests for url: https://huggingface.co/api/models/Qdrant/bm25 (Amz CF ID: pCeO0-nFgukrWKYmyuAbfvWNuIa6bWX15D-EDjfvsuDexPwO8JknQQ==) Falling back to other sources.
2026-04-05 05:11:56.929 | ERROR    | fastembed.common.model_management:download_model:469 - Could not download model from either source, sleeping for 3.0 seconds, 2 retries left.
2026-04-05 05:11:59.940 | ERROR    | fastembed.common.model_management:download_model:446 - Could not download model from HuggingFace: 429 Client Error: Too Many Requests for url: https://huggingface.co/api/models/Qdrant/bm25 (Amz CF ID: C-ztBctg5H9jODzj4w03S_41R9qmMHWsVkjQDKRXxU1BPElM1qvdYg==) Falling back to other sources.
2026-04-05 05:11:59.941 | ERROR    | fastembed.common.model_management:download_model:469 - Could not download model from either source, sleeping for 9.0 seconds, 1 re

KeyboardInterrupt: 

# Qdrant向量存储

In [8]:
from rag_modules import DataPreparationModule

In [33]:
data_module = DataPreparationModule("../../data/C8/cook")
data_module.load_documents()
data_module.chunk_documents();

In [12]:
data_module.chunks[0].metadata

{'主标题': '小酥肉的做法',
 'source': '../../data/C8/cook/dishes/meat_dish/小酥肉.md',
 'parent_id': 'a8d62aca1273a4da20cab6ab4cf25c1d',
 'doc_type': 'child',
 'category': '荤菜',
 'dish_name': '小酥肉',
 'difficulty': '中等',
 'chunk_id': '6509eebc-ab22-4066-80ef-68e11fed5511',
 'chunk_index': 0,
 'batch_index': 0,
 'chunk_size': 21}

In [16]:
# 构建稠密、稀疏向量
chunk_texts = [chunk.page_content for chunk in data_module.chunks][:2]
dense_vecs = await dense_model.aembed_documents(chunk_texts)
sparse_vecs = sparse_model.embed_batch(chunk_texts)

In [25]:
# 构建数据点
points = []
for dense_vec, sparse_vec, chunk in zip(dense_vecs, sparse_vecs, data_module.chunks):
    point = PointStruct(
        id = chunk.metadata['chunk_id'], 
        vector={
            'dense': dense_vec, 
            'sparse': SparseVector(
                indices=sparse_vec.indices.tolist(), # toekn对应的索引
                values=sparse_vec.values.tolist(),   # 索引对应的token的得分
            )
        }, 
        payload=chunk.metadata
    )
    points.append(point)

In [30]:
await qdrant_client.upsert(
    collection_name=chunk_collection_name, 
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

# Qdrant检索

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from rag_modules.data_preparation import DataPreparationModule

import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional, Literal

load_dotenv()

/opt/conda/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


True

In [2]:
model_name, temperature = os.getenv('MOONSHOT_MODEL_FAST'), 0.4  # 允许自由发挥，但不能过多
fast_llm = init_chat_model(
    model=model_name, 
    model_provider='openai',
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
    temperature=temperature 
)

# llm = init_chat_model(
#     model=os.getenv('MOONSHOT_MODEL_ID'), 
#     model_provider='openai',
#     api_key=os.getenv("MOONSHOT_API_KEY"),
#     base_url=os.getenv("MOONSHOT_BASE_URL"),
#     temperature=0.1
# )

## HyDE 假设性文档嵌入

In [5]:
# query = "今天气温很低，外面下雪，你推荐一些菜，注意我不太喜欢吃辣"
query = "世界杯开赛了，我要半夜看球，想在看球时候吃一些东西，你推荐一下。"
# query = "我最近查出来血糖高，我也不太想做饭，你推荐一些菜吧？"

In [6]:
hyde_cot_prompt = f"""
你是一个顶级的 AI 饮食推荐引擎。你的任务是深度解析用户的饮食需求，进行意图推理后，生成**三个**相关的、极具画面感的菜品描述（假设性菜谱文档），用于后续的向量数据库检索。

【生成要求】
1. 必须输出且仅输出一个合法的 JSON 对象，不要包含任何多余的解释文字或 Markdown 标记符号（如 ```json）。
2. JSON 对象必须包含以下两个字段：
   - "analyze": (字符串) 你的思考过程。简要分析用户的真实意图、情绪场景、所需的营养搭配，以及推导应该采用哪三种不同的烹饪手法/食材组合来满足用户。
   - "hyde_documents": (JSON数组, 数组内是3个字符串) 包含 3 个不同的菜谱描述。每个描述 50-100 字。
3. ⚠️ 对于 hyde_documents 的极其重要要求：描述中要大量使用【烹饪动词】（如：大火爆香、慢炖、收汁）、【感官形容词】（如：外酥里嫩、汤汁浓郁、清脆解腻）和【具体食材类别】。模仿真实菜谱的正文口吻，绝对不要写干瘪的标签。

【Few-Shot 示例】

输入：下班好累，想做个有肉的快手菜，最好十分钟搞定。
输出：
{{
    "analyze": "用户当前状态是疲惫，核心诉求是‘有肉’和‘极度快手(10分钟内)’。为了满足需求，我需要避开耗时的炖煮，选择三种最高效的烹饪策略：1. 猛火爆炒薄肉片；2. 微波炉或蒸锅免洗锅做法；3. 平底锅高温干煎肉排。",
    "hyde_documents": [
        "热锅冷油，将腌制好的肉片大火快速滑炒至变色。加入葱姜蒜爆香，淋入生抽和少许老抽上色，翻炒均匀。整个过程只需几分钟，肉质鲜嫩多汁，浓郁的酱汁极其下饭，是一道极其抚慰人心的快手爆炒肉菜。",
        "不需要复杂的起锅烧油，将切成薄片的肉类和洗净的快熟蔬菜码放在深盘中，淋上调配好的葱油或蒜蓉酱汁。直接放入微波炉高温加热几分钟，或者上蒸锅大火速蒸。肉质软嫩入味，汤汁拌饭绝佳，做法极度省事且毫无油烟。",
        "平底锅大火烧热，直接将带少许脂肪的肉块或肉排下锅，煎至两面金黄微焦，逼出多余油脂。只撒上简单的海盐和现磨黑胡椒提味。外皮酥脆，内里鲜嫩爆汁，做法粗犷快手，大口吃肉极大缓解了一天的疲惫。"
    ]
}}

输入：最近在减脂期，想吃点清淡低卡的，但要有饱腹感。
输出：
{{
    "analyze": "核心诉求是‘减脂低卡’和‘高饱腹感’。低卡意味着烹饪要少油无油（如凉拌、清汤、蒸煮），高饱腹感则需要优质蛋白（白肉、豆制品）和高纤维（粗粮、菌菇）。我将提供凉拌高蛋白、清透鲜蔬汤和粗粮蒸煮三种方向的描述。",
    "hyde_documents": [
        "这是一道清爽解腻的低卡凉拌菜。将富含优质高蛋白的低脂白肉水煮断生后撕成细丝，搭配大量清脆的高纤维蔬菜（如黄瓜、木耳）。淋上由柠檬汁、生抽和少许代糖调制的无油醋汁。口感酸甜开胃，咀嚼感极强，吃一大盘也毫无罪恶感。",
        "一道热气腾腾的低脂鲜蔬汤。砂锅中加入清水，放入滑嫩的豆腐、菌菇和各种时令绿叶菜一起大火煮沸。全程不滴一滴明油，只用少许盐和白胡椒粉简单调味。汤汁清透鲜美，菌菇自带氨基酸提鲜。热乎乎地下肚，既暖胃又能提供极强的饱腹感。",
        "将富含优质碳水的根茎类粗粮（如红薯、南瓜）与低脂高蛋白食材一起上蒸锅清蒸。这种烹饪方式保留了食物最原始的清甜与本味。出锅后只需蘸取少许清淡的蘸水食用。少油少盐，营养比例完美，饱腹感持久，是非常优秀的减脂期正餐替代品。"
    ]
}}

输入：{query}
输出：
"""

In [9]:
response = fast_llm.invoke([SystemMessage(content=hyde_cot_prompt)])

hyde_recipes = json.loads(response.content)  # json对象 -> dict对象
# for hyde_recipe in hyde_recipes:
#     print(hyde_recipe)
#     print()

In [13]:
hyde_recipes

{'analyze': '用户深夜看球，情绪兴奋，需要“零打扰、单手能吃、耐放不冷”的零食级硬菜。核心诉求是重口解馋、抗饿、不脏手。我将用三种“提前做→室温仍好吃”的思路：1. 先卤后炸的酥香小件；2. 干香腊味空气炸；3. 浓酱裹炸的吮指排骨。',
 'hyde_documents': ['先将鸡胗、鸭舌等小件冷水下锅，加八角、桂皮、香叶、老抽慢卤四十分钟入味，捞出沥干；油锅烧至七成热，下卤料大火复炸三十秒，外壳瞬间起泡收缩，撒辣椒面、花椒粉、孜然粒翻匀，入口先是咔嚓脆壳，紧接着卤汁爆浆，筋道耐嚼，看球时一颗接一颗完全停不下来。',
  '广式腊肠斜刀切成厚片，空气炸锅200℃预热后平铺入锅，六分钟逼出晶莹油脂，肠衣收缩出油亮红光，翻面再炸三分钟，表面形成焦糖色脆壳，内部甘甜酒香浓缩；趁热撒少许熟白芝麻，脂香四溢，直接用手捏着吃，不掉渣不脏遥控，越嚼越上头。',
  '肋排小段冷水去血沫后，用姜末、料酒、黄豆酱、少许蜂蜜腌半小时；油锅中火炸至表面金黄锁住肉汁，另起炒锅下蒜末、干辣椒段小火煸香，倒入排骨大火翻炒收汁，酱汁浓稠裹匀，撒葱花与熟芝麻提亮；放至室温依然外酥里嫩，手指一捏脱骨，吮指回味与进球欢呼完美同步。']}

## 自检索器

In [ ]:
import qdrant_client.http.models as models

In [15]:
class RecipeMetadataParse(BaseModel):
    """
    菜谱元数据解析器
    用于将用户中的提问转变为元数据信息
    """
    dish_name: Optional[str] = Field(default=None, description='菜品名称')
    category: Optional[str] = Field(default=None, description='菜品类别')
    difficulty: Optional[str] = Field(default=None, description='菜品制作难度')

    @classmethod
    def get_options(cls) -> list[str]:
        """返回对元数据字段限制性描述"""
        desc =  [
            f"category 字段只能属于以下列表{DataPreparationModule.CATEGORY_LABELS}，请不要随意编撰", 
            f"difficulty 字段只能属于以下列表{DataPreparationModule.DIFFICULTY_LABELS}，请不要随意编撰", 
        ]
        return desc

In [53]:
structure_llm = fast_llm.with_structured_output(RecipeMetadataParse)
structure_llm.get_output_jsonschema()

{'description': '菜谱元数据解析器\n用于将用户中的提问转变为元数据信息',
 'properties': {'dish_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品名称',
   'title': 'Dish Name'},
  'category': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品类别',
   'title': 'Category'},
  'difficulty': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': '菜品制作难度',
   'title': 'Difficulty'}},
 'title': 'RecipeMetadataParse',
 'type': 'object'}

In [58]:
query = "能推荐几个好做的早餐吗？"

metadata_parser_prompt = f"""你是一个专业的信息解析员，你的职责就是提取 用户提问 中的相关信息，并将所需字段填写到下面的object中
    
注意：
 - 用户输入中没有出现的字段决不能主观猜测，返回None即可，不能随便填入
 {'- ' + '\n - '.join(RecipeMetadataParse.get_options())}

用户提问：{query}"""
    
hard_fliters = structure_llm.invoke([SystemMessage(content=metadata_parser_prompt)])

In [59]:
hard_fliters.model_dump()

{'dish_name': None, 'category': '早餐', 'difficulty': None}

models.FieldCondition(

    *,
    
    key: str,
    
    match: Union[qdrant_client.http.models.models.MatchValue, qdrant_client.http.models.models.MatchText, qdrant_client.http.models.models.MatchTextAny, qdrant_client.http.models.models.MatchPhrase, qdrant_client.http.models.models.MatchAny, qdrant_client.http.models.models.MatchExcept, NoneType] = None,
    
    range: Union[qdrant_client.http.models.models.Range, qdrant_client.http.models.models.DatetimeRange, NoneType] = None,
    
    geo_bounding_box: Optional[qdrant_client.http.models.models.GeoBoundingBox] = None,
    
    geo_radius: Optional[qdrant_client.http.models.models.GeoRadius] = None,
    
    geo_polygon: Optional[qdrant_client.http.models.models.GeoPolygon] = None,
    
    values_count: Optional[qdrant_client.http.models.models.ValuesCount] = None,
    
    is_empty: Optional[bool] = None,
    
    is_null: Optional[bool] = None,
    
) -> None

In [ ]:
# hard_filters是llm返回的从用户询问中提取到的元数据

must_conditions = []  # 菜谱该条元数据必须满足该条件，也就是只选择满足条件的菜谱
must_not_conditions = []  # 菜谱该条元数据不能等于该条件，也就是排除符合条件的菜谱

if hard_filters.get('category'):
    must_conditions.append(
        models.FieldCondition(
            key='category',  # 精准匹配菜品类型
            match=models.MatchAny(any=hard_filters['category'])
        )
    )
    
if hard_filters.get('origin'):
    must_conditions.append(
        models.FieldCondition(
            key='origin', # 精准匹配菜品发源地
            match=models.MatchAny(any=hard_filters['origin'])
        )
    )

if hard_filters.get('max_spiciness'):
    must_conditions.append(
        models.FieldCondition(
            key="spiciness", # 辣度值
            range=models.Range(lte=hard_filters["max_spiciness"])  # 范围查询，小于等于
        )
    )

if hard_filters.get('exclude_ingredients'):
    must_not_conditions.append(
        models.FieldCondition(
            key="ingredients",
            # 注意：这里放在 must_not 里面，配合 MatchAny，
            # 意思就是“Payload 的 ingredients 数组中，绝对不能出现这几个词中的任意一个”
            match=models.MatchAny(any=hard_filters["exclude_ingredients"])
        )
    )
# ...还有甜度、酸度、苦、等一系列可以添加的元数据

In [7]:
# 将条件组装成最终的 Qdrant Filter 对象
recipe_filter = models.Filter(
    must=must_conditions,
    must_not=must_not_conditions
)

In [12]:
search_result = qdrant_client.search(
    collection_name=collection_name,
    query_vector=user_query,    # 向量查询，这里也可以使用混合检索
    query_filter=recipe_filter, # 元数据过滤器
    limit=3,                    # 只取 Top 3 最匹配的
    with_payload=True           # 要求 Qdrant 把完整的元数据也一并返回给你
)

## 混合检索

In [1]:
from FlagEmbedding import FlagReranker
reranker_model = FlagReranker(
    model_name_or_path='BAAI/bge-reranker-v2-m3', 
    use_fp16=True, 
    cache_dir='/tmp/reranker-model/'
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [27]:
query = '推荐一些热量高的菜？'
query_dense_vec = (await dense_model.aembed_documents([query]))[0]
# query_dense_vec = dense_model.embed_query(query)
query_sparse_vec = sparse_model.embed_single(query)

In [30]:
results = await qdrant_client.query_batch_points(
    collection_name=chunk_collection_name, 
    requests=[
        models.QueryRequest(
            using="dense",
            query=query_dense_vec,
            limit=10, 
            with_payload=['parent_id']
        ), 
        models.QueryRequest(
            using='sparse', 
            query=SparseVector(
                indices=query_sparse_vec.indices.tolist(),
                values=query_sparse_vec.values.tolist()
            ), 
            limit=10, 
            with_payload=['parent_id']
        )
    ], 
)

In [40]:
vector_ids = list(point.payload['parent_id'] for point in results[0].points if point.payload.get('parent_id'))  # 未去重
bm25_ids = list(point.payload['parent_id'] for point in results[1].points if point.payload.get('parent_id'))

vector_docs = data_module.get_parent_documents(vector_ids)
bm25_docs = data_module.get_parent_documents(bm25_ids)

for chunk in vector_docs:
    print(chunk.metadata['dish_name'])
print('=' * 10)
for chunk in bm25_docs:
    print(chunk.metadata['dish_name'])

回锅肉
水油焖蔬菜
奶酪培根通心粉
水煮肉片
辣椒炒肉
山西过油肉
洋葱炒猪肉
小米辣炒肉
茭白炒肉
小米粥
猪皮冻
糖拌西红柿
蛋煎糍粑
麻辣香锅
昂刺鱼豆腐汤
鲤鱼炖白菜
示例菜
芹菜拌茶树菇
白灼菜心


In [48]:
def rrf_rerank(vector_docs, bm25_docs, k: int = 60, weight=[0.6, 0.4]):
    """
    使用RRF (Reciprocal Rank Fusion) 算法重排文档

    Args:
        vector_docs: 向量检索结果
        bm25_docs: BM25检索结果
        k: RRF参数，用于平滑排名

    Returns:
        重排后的文档列表
    """
    doc_scores = {}
    doc_objects = {}

    # 计算向量检索结果的RRF分数
    for rank, doc in enumerate(vector_docs):
        # 使用文档内容的哈希作为唯一标识
        doc_id = hash(doc.page_content)
        doc_objects[doc_id] = doc

        # RRF公式: 1 / (k + rank)
        rrf_score = weight[0] * 1.0 / (k + rank + 1)
        doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

        print(f"向量检索 - 文档{rank+1}: RRF分数 = {rrf_score:.4f}")

    # 计算BM25检索结果的RRF分数
    for rank, doc in enumerate(bm25_docs):
        doc_id = hash(doc.page_content)
        doc_objects[doc_id] = doc

        rrf_score = weight[1] * 1.0 / (k + rank + 1)
        doc_scores[doc_id] = doc_scores.get(doc_id, 0) + rrf_score

        print(f"BM25检索 - 文档{rank+1}: RRF分数 = {rrf_score:.4f}")

    # 按最终RRF分数排序
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

    # 构建最终结果
    reranked_docs = []
    for doc_id, final_score in sorted_docs:
        if doc_id in doc_objects:
            doc = doc_objects[doc_id]
            # 将RRF分数添加到文档元数据中
            doc.metadata['rrf_score'] = final_score
            reranked_docs.append(doc)

    print(f"RRF重排完成: 向量检索{len(vector_docs)}个文档, BM25检索{len(bm25_docs)}个文档, 合并后{len(reranked_docs)}个文档")
   
    return reranked_docs

In [50]:
res_docs = rrf_rerank(vector_docs, bm25_docs);

向量检索 - 文档1: RRF分数 = 0.0098
向量检索 - 文档2: RRF分数 = 0.0097
向量检索 - 文档3: RRF分数 = 0.0095
向量检索 - 文档4: RRF分数 = 0.0094
向量检索 - 文档5: RRF分数 = 0.0092
向量检索 - 文档6: RRF分数 = 0.0091
向量检索 - 文档7: RRF分数 = 0.0090
向量检索 - 文档8: RRF分数 = 0.0088
向量检索 - 文档9: RRF分数 = 0.0087
BM25检索 - 文档1: RRF分数 = 0.0066
BM25检索 - 文档2: RRF分数 = 0.0065
BM25检索 - 文档3: RRF分数 = 0.0063
BM25检索 - 文档4: RRF分数 = 0.0063
BM25检索 - 文档5: RRF分数 = 0.0062
BM25检索 - 文档6: RRF分数 = 0.0061
BM25检索 - 文档7: RRF分数 = 0.0060
BM25检索 - 文档8: RRF分数 = 0.0059
BM25检索 - 文档9: RRF分数 = 0.0058
BM25检索 - 文档10: RRF分数 = 0.0057
RRF重排完成: 向量检索9个文档, BM25检索10个文档, 合并后19个文档


In [51]:
for doc in res_docs:
    print(doc.metadata['dish_name'])

回锅肉
水油焖蔬菜
奶酪培根通心粉
水煮肉片
辣椒炒肉
山西过油肉
洋葱炒猪肉
小米辣炒肉
茭白炒肉
小米粥
猪皮冻
糖拌西红柿
蛋煎糍粑
麻辣香锅
昂刺鱼豆腐汤
鲤鱼炖白菜
示例菜
芹菜拌茶树菇
白灼菜心


In [29]:
res = await qdrant_client.query_points(
    collection_name=chunk_collection_name, 
    prefetch=[
        Prefetch(
            using='dense', 
            query=query_dense_vec,
            limit=10
        ),
        Prefetch(
            using='sparse', 
            query=SparseVector(
                indices=query_sparse_vec.indices.tolist(),
                values=query_sparse_vec.values.tolist()
            ), 
            limit=10
        )
    ], 
    query=FusionQuery(fusion=Fusion.RRF), 
    limit=10
)

In [31]:
text_pairs = []
for doc in candidate_docs:
    text_pairs.append((query, doc.page_content))
scores = reranker_model.compute_score(text_pairs)
scores = [(score, i) for i, score in enumerate(scores)]
scores.sort(reverse=True, key=(lambda x: x[0]))

In [32]:
reranked_docs = []
count = 0
for score, seq in scores:
    # if score < 0:     # 这里设置一个排序模型的分数阈值，只有当文档与查询的相关性得分超过threshold时才被认为是相关的，才会被加入到重排结果中
    #     continue
    reranked_docs.append(candidate_docs[seq])
    count += 1

In [33]:
for chunk in reranked_docs:
    print(chunk.metadata['dish_name'])

鱼香茄子
蒲烧茄子
炒茄子
烤茄子
牛排
示例菜
英式司康


In [34]:
scores

[(-1.041517972946167, 1),
 (-1.3858991861343384, 4),
 (-1.9054242372512817, 2),
 (-3.054213523864746, 0),
 (-4.4638237953186035, 3),
 (-5.051511764526367, 5),
 (-6.526769638061523, 6)]

# Other